# Лабораторная работа №1 — разведочный анализ данных

Разведочный анализ подготовленной Iceberg-таблицы средствами Apache Spark.

Основные этапы:

- анализ структуры;
- исследование пропусков;
- описательные статистики;
- категориальные распределения;
- квантили и IQR;
- поиск выбросов;
- распределения числовых признаков;
- корреляционный анализ.

## 1) Инициализация

In [14]:
%matplotlib inline

from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.sql.functions import (
    col,
    count,
    desc,
    floor,
    isnan,
    max as spark_max,
    min as spark_min,
    sum as spark_sum,
    when,
)

from src.spark_session import create_spark

spark = create_spark("Lab_1_data_analysis")

TABLE_NAME = "local.lab1.steam_games"

RESULTS_DIR = Path("/app/output/results")
PLOTS_DIR = Path("/app/output/plots")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## 2) Загрузка и просмотр очищенных данных

In [15]:
df = spark.table(TABLE_NAME)

print("Количество строк:", f"{df.count():,}")
print("Количество столбцов:", len(df.columns))

df.printSchema()


Количество строк: 140,243
Количество столбцов: 24
root
 |-- app_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- recommendations: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- metacritic_score: integer (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- achievements: integer (nullable = true)
 |-- dlc_count: integer (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- estimated_owners

In [16]:
df.orderBy("app_id", truncate=False).show()
#df.show(10, truncate=False)

+------+--------------------+------------+-----+--------+--------------------+--------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+-----+-----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|app_id|                name|release_date|price|  genres|          categories|                tags|positive|negative|recommendations|peak_ccu|metacritic_score|average_playtime_forever|median_playtime_forever|achievements|dlc_count|windows|  mac|linux|estimated_owners_min|estimated_owners_max|estimated_owners_avg|supported_languages_count|full_audio_languages_count|
+------+--------------------+------------+-----+--------+--------------------+--------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+-----+-----+-------------

Узнаем сколько значений null есть в каждом столбце

In [17]:
#from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df.select(
    [
        spark_sum(
            when(col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in df.columns
    ]
)

null_counts.show(truncate=False)

+------+----+------------+-----+------+----------+----+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+---+-----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|app_id|name|release_date|price|genres|categories|tags|positive|negative|recommendations|peak_ccu|metacritic_score|average_playtime_forever|median_playtime_forever|achievements|dlc_count|windows|mac|linux|estimated_owners_min|estimated_owners_max|estimated_owners_avg|supported_languages_count|full_audio_languages_count|
+------+----+------------+-----+------+----------+----+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------+---+-----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|0     |16  |83          |1195 |0 

In [19]:
from src.spark_session import create_spark

from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    when,
)


# Инициализация Spark
spark = create_spark("CheckRawCsvNulls")


# Путь к исходному CSV
INPUT_PATH = "/app/data/raw/steam_games.csv"


# Чтение исходного CSV без каких-либо преобразований
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(INPUT_PATH)
)


# Если первый столбец имеет пустое имя
first_column = df_raw.columns[0]

if first_column != "app_id":
    df_raw = df_raw.withColumnRenamed(
        first_column,
        "app_id",
    )


print("Количество строк:", df_raw.count())
print("Количество столбцов:", len(df_raw.columns))


# Подсчёт NULL по каждому столбцу
null_counts = df_raw.select(
    [
        spark_sum(
            when(
                col(column).isNull(),
                1,
            ).otherwise(0)
        ).alias(column)
        for column in df_raw.columns
    ]
)


null_counts.show(
    truncate=False,
    vertical=True,
)

Количество строк: 140243
Количество столбцов: 42


26/09/12 18:16:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0--------------------------
 app_id                   | 0      
 name                     | 16     
 release_date             | 83     
 price                    | 1195   
 price_status             | 0      
 estimated_owners         | 0      
 developers               | 0      
 publishers               | 0      
 genres                   | 0      
 categories               | 0      
 tags                     | 0      
 positive                 | 0      
 negative                 | 0      
 recommendations          | 33049  
 peak_ccu                 | 0      
 metacritic_score         | 40162  
 user_score               | 0      
 average_playtime_forever | 0      
 median_playtime_forever  | 0      
 average_playtime_2weeks  | 0      
 median_playtime_2weeks   | 0      
 short_description        | 8331   
 about_the_game           | 8438   
 detailed_description     | 8417   
 notes                    | 114026 
 achievements             | 17477  
 dlc_count                | 